# 07 · Подготовка данных: нейтрализация аномальных периодов

**Цель** — построить обучающую панель «канал × день» с признаками до дня t
и целью «отказ в +24ч», при этом **аномальный участок (кампании проверок)
не должен искажать признаки и разметку**.

Ключевой вопрос: что делать с аномальным периодом?
- **Медианная замена** (предпочтение команды): в аномальные дни признаки
  заменяются на «типичный фон» канала (медиана чистых дней), чтобы
  скользящие статистики не впитали всплеск.
- Аномальные дни **не размечаются** как «отказ» (в цель ставки не идёт).

Опирается на `data_utils.CAMPAIGN_WEEKS` и результаты EDA (ноутбук 06).

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import numpy as np
import data_utils as du

# True — полный пересчёт агрегатов из сырых CSV (медленно, минуты-часы).
# False — читать готовые артефакты dataset/daily_agg_*.csv.
RECOMPUTE = False

print("dataset:", du.find_dataset_dir())

# 7.1 · Как аномалия влияет на типы датчиков

Проверяем по готовому `weekly_alarms_by_type.csv`: доля тревог в аномальные
недели от всех тревог типа. Ожидаем: **дым — подавляющее большинство**,
остальные типы — фоновый шум.

In [ ]:
def _is_anomaly(wk: int) -> bool:
    return du.week_is_campaign(int(wk))

at = pd.read_csv(du.find_dataset_dir() / "weekly_alarms_by_type.csv",
                 dtype={"год_неделя": int})
anom = at["год_неделя"].map(_is_anomaly)
impact = at.groupby("тип_датчика").apply(
    lambda g: pd.Series({
        "всего": g["тревог"].sum(),
        "аномально": g.loc[g["год_неделя"].map(_is_anomaly), "тревог"].sum(),
    }), include_groups=False).reset_index()
impact["доля_аном_%"] = (100 * impact["аномально"] / impact["всего"]).round(1)
impact = impact.sort_values("аномально", ascending=False)

print("Влияние аномальных недель по типам (топ-15):")
print(impact.head(15).to_string(index=False))

print("\nВывод: критично затронут Дым (доля > 80%); остальные < 5% → фон.")
fault_types = impact[impact["доля_аном_%"] > 5]["тип_датчика"].tolist()
print("Типы, требующие нейтрализации:", fault_types)

# 7.2 · Построение суточной панели (канал × день)

Для выбранного диапазона считаем агрегаты на (ид_канала_данных, дата):
- `событий`, `тревог`, `неисправностей` (статусы «Неисправен/Обесточен/Отключено устройство»),
- `шума` (значения 0.00 / 0.01 / смены состояния),
- медиану числового значения (для телеметрии).

Артефакт: `dataset/daily_agg_{год}_{тип}.csv`.

In [ ]:
from pathlib import Path

# Какие типы обрабатываем (первый прогон — дым, самый пострадавший)
TARGET_TYPES = ["Датчик дыма"]
FAULT_STATUSES = {"Неисправен", "Обесточен", "Отключено устройство"}
YEARS_ALL = list(map(str, range(2019, 2027)))
DATASET = du.find_dataset_dir()
CHUNK = 1_000_000


def build_daily_for_type(typ: str, years, out_csv: Path, compute_median: bool = False):
    """Суточные агрегаты (канал × дата) для одного типа датчиков.

    Векторно: по чанку делаем groupby(канал, дата).agg(...), кусочки года
    копим в список, в конце года пишем CSV. Пик памяти ~ размер года.
    """
    ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика"]]
    ch_set = set(ref_ch.loc[ref_ch["тип_датчика"] == typ, "ид_канала_данных"])

    year_files = []
    for fp in du.journal_files():
        year = fp.name.split("-")[2].split(".")[0]
        if year not in years:
            continue
        year_parts = []
        for ch in du.read_chunks(fp,
                                 cols=["ид_канала_данных", "дата", "тревожное", "значение_датчика"],
                                 chunksize=CHUNK):
            m = ch["ид_канала_данных"].isin(ch_set)
            if not m.any():
                continue
            ch = ch[m]
            g = ch.groupby(["ид_канала_данных", "дата"], sort=False)
            agg = pd.DataFrame({
                "событий": g.size(),
                "тревог": g["тревожное"].apply(lambda s: (s == "t").sum()),
                "неисправностей": g["значение_датчика"].apply(
                    lambda s: s.isin(FAULT_STATUSES).sum()),
                "шума": g["значение_датчика"].apply(
                    lambda s: s.isin(["0.00", "0.01", "0.02"]).sum()),
            }).reset_index()
            if compute_median:
                vals = ch.assign(vnum=pd.to_numeric(
                    ch["значение_датчика"].str.replace(",", "."), errors="coerce"))
                med = (vals.groupby(["ид_канала_данных", "дата"])["vnum"]
                           .median().rename("медиана_знач"))
                agg = agg.merge(med, on=["ид_канала_данных", "дата"], how="left")
            year_parts.append(agg)
        pdf = pd.concat(year_parts, ignore_index=True)
        yf = out_csv.with_name(f"daily_agg_{typ.split()[-1]}_{year}.csv")
        pdf.to_csv(yf, index=False, encoding="utf-8-sig")
        year_files.append(yf)
        del year_parts, pdf
        print(f"[{typ}::{year}] ок, строк={pd.read_csv(yf).shape[0]:,}")

    # объединение годовых файлов
    df = pd.concat([pd.read_csv(f, dtype={"ид_канала_данных": str, "дата": str})
                    for f in year_files], ignore_index=True)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[{typ}] панель: {out_csv.name}, строк: {len(df):,}")

for typ in TARGET_TYPES:
    key = "дым" if "дым" in typ.lower() else typ.replace(" ", "_").lower()
    out_csv = DATASET / f"daily_agg_{YEARS_ALL[0]}_{YEARS_ALL[-1]}_{key}.csv"
    if RECOMPUTE or not out_csv.exists():
        build_daily_for_type(typ, YEARS_ALL, out_csv,
                             compute_median=("температура" in typ.lower()))
    else:
        print("артефакт уже есть:", out_csv.name)

# 7.3 · Нейтрализация аномальных периодов (медианная замена)

Для каждого канала в аномальные дни (из `CAMPAIGN_WEEKS`) заменяем
признаки на **медиану «чистых» дней этого канала**. Цель: скользящие
фичи не впитают всплеск, канал не «объявится отказавшим» из-за проверки.

Логика (почему медиана, а не NaN):
- LightGBM умеет NaN, но скользящие статистики (mean/trend за 7/30д)
  «загрязняются» или рвутся; медианная замена сохраняет фон канала.
- Медиана считается **по чистым дням канала** (вне аномалий), т.е.
  «типичное поведение», а не среднее по загрязнённому периоду.

После замены строим:
- `цель_24ч` — было ли событие неисправности в следующие 24 часа (по чистым дням);
- аномальные дни получают `аномально=True` и **исключаются из обучения**.

In [ ]:
daily = pd.read_csv(du.find_dataset_dir() / "daily_agg_2019_2026_дым.csv",
                    dtype={"ид_канала_данных": str, "дата": str})

# ISO-неделя из даты
iso = pd.to_datetime(daily["дата"]).dt.isocalendar()
daily["год_неделя"] = iso["year"].astype(int) * 100 + iso["week"].astype(int)
daily["аномально"] = daily["год_неделя"].map(du.week_is_campaign)

print("Строк панели:", len(daily), "| аномальных дней:", int(daily["аномально"].sum()))

# Медиана по чистым дням канала (для замены)
FEAT_BASE = ["событий", "тревог", "неисправностей", "шума"]
feat_cols = [c for c in FEAT_BASE + ["медиана_знач"] if c in daily.columns]
clean = daily[~daily["аномально"]]
ch_med = clean.groupby("ид_канала_данных")[feat_cols].median()

# Замена аномальных дней на медиану канала (векторно через map)
daily_filled = daily.copy()
# pandas 3.0 строг к типам: заранее приводим признаки к float, чтобы вписать медианы
daily_filled[feat_cols] = daily_filled[feat_cols].astype(float)
anom_mask = daily_filled["аномально"].values
ch_med_map = ch_med.reindex(daily_filled["ид_канала_данных"].unique()).fillna(
    clean[feat_cols].median())  # fallback для каналов без чистых дней
vals = ch_med_map.loc[daily_filled["ид_канала_данных"].values].values
daily_filled.loc[anom_mask, feat_cols] = vals[anom_mask]

print("\nДо/после (пример аномального дня, канал с большим всплеском):")
big = daily.loc[daily["аномально"], "событий"].idxmax()
print("аномальный день, канал:", daily.loc[big, "ид_канала_данных"],
      daily.loc[big, "дата"], "было:", daily.loc[big, feat_cols].round(2).tolist())
print("стало:", daily_filled.loc[big, feat_cols].round(2).tolist())
print("медиана канала:", ch_med.loc[daily.loc[big, "ид_канала_данных"], feat_cols].round(2).tolist())

# 7.4 · Скользящие признаки и целевая переменная

На заполненной (нейтрализованной) панели считаем для каждого канала:
- скользящие суммы событий/тревог/неисправностей за 3, 7, 14, 30 дней;
- z-скор частоты относительно медианы канала;
- целевую переменную **цель_24ч**: появилось ли «неисправность/отказ»
  в канале в течение следующих 24 часов.

Аномальные дни → исключаются из обучения (`аномально=False`), но остаются
в панели, чтобы скользящие окна «нормально» проходили сквозь период.

In [ ]:
# Сортировка по каналу и дате
daily_filled = daily_filled.sort_values(["ид_канала_данных", "дата"]).reset_index(drop=True)
daily_filled["дата_dt"] = pd.to_datetime(daily_filled["дата"])

f = daily_filled.groupby("ид_канала_данных", sort=False)
for w in (3, 7, 14, 30):
    daily_filled[f"событий_{w}д"] = f["событий"].transform(
        lambda s: s.rolling(w, min_periods=1).sum())
    daily_filled[f"неисправностей_{w}д"] = f["неисправностей"].transform(
        lambda s: s.rolling(w, min_periods=1).sum())
    daily_filled[f"тревог_{w}д"] = f["тревог"].transform(
        lambda s: s.rolling(w, min_periods=1).sum())

# z-скор частоты относительно медианы канала (по чистым дням)
ch_med_events = clean.groupby("ид_канала_данных")["событий"].median()
ch_iqr = clean.groupby("ид_канала_данных")["событий"].quantile(0.75) - \
         clean.groupby("ид_канала_данных")["событий"].quantile(0.25)
daily_filled["z_событий"] = (daily_filled["событий"] - daily_filled["ид_канала_данных"].map(ch_med_events)) / \
                            (daily_filled["ид_канала_данных"].map(ch_iqr) + 1e-6)

# Целевая переменная: неисправность в следующие 24 часа (по исходным, НЕ замененным данным)
target = daily.sort_values(["ид_канала_данных", "дата"]).reset_index(drop=True)
t = target.groupby("ид_канала_данных", sort=False)
# сдвигаем флаг неисправности вверх на 1 день → «была неисправность в следующие 24ч»
daily_filled["цель_24ч"] = t["неисправностей"].transform(
    lambda s: s.shift(-1).fillna(0).gt(0).astype(int))
# аномальные дни исключаем из обучения (цель обесцениваем)
daily_filled.loc[daily_filled["аномально"], "цель_24ч"] = np.nan

print("Доля положительных (по чистым дням):",
      round(daily_filled["цель_24ч"].mean() * 100, 3), "%")
print(daily_filled["цель_24ч"].value_counts(dropna=False).to_dict())

# 7.5 · Сохранение готовой панели

Сохраняем нейтрализованную панель для обучения (артефакт
`dataset/train_panel_дым.csv`). Колонка `аномально` позволяет исключить
проблемные дни из сплитов, а `цель_24ч` — метка отказа на 24ч вперёд.

In [ ]:
out_panel = du.find_dataset_dir() / "train_panel_дым.csv"
daily_filled.to_csv(out_panel, index=False, encoding="utf-8-sig")
print("Сохранено:", out_panel.name, "| строк:", len(daily_filled))

# Итоговый состав
keep = ["ид_канала_данных", "дата", "год_неделя", "аномально", "цель_24ч"] + feat_cols + \
       [f"{c}_{w}д" for c in ("событий", "неисправностей", "тревог") for w in (3, 7, 14, 30)] + \
       ["z_событий"]
print("Колонки:", [c for c in keep if c in daily_filled.columns])

# 7.6 · Комментарии и выбор стратегии

**Итог:**
- Аномальные недели (2021W16–W23, 2026W23) затронули почти только **дым**
  (90.6% тревог дыма за 8 лет), остальные типы — фон < 5%.
- Для дыма построена панель «канал × день» с **медианной заменой** аномальных
  дней и целью «отказ в следующие 24ч».
- Аномальные дни помечены и **исключаются из обучения**, но остаются в панели,
  чтобы скользящие окна проходили непрерывно.

**Почему медиана (а не NaN/пропуск строк):**
- ГБМ (LightGBM) терпит NaN, но скользящие фичи рвутся/загрязняются;
- удаление строк аномалии создаёт «дыры» в последовательности канала,
  из-за чего окна после аномалии считаются от обрезанной истории;
- медиана **чистых дней канала** даёт «типичное поведение» — канал
  не выбивается ни в плюс (всплеск проверки), ни в минус (пропадание).

**Ограничения:**
- Медианная замена — это про «нейтральность», не про «точную реконструкцию»;
- если в реальных данных будут длительные «выключения» канала (не кампании),
  их надо обрабатывать отдельно (флаг no_data), а не медианой.